# Breast Cancer Subtype Prediction
# Step 3: Feature Selection - Phase 1 (Variance Threshold)
# Goal: Remove constant and quasi-constant features from RNA & Methylation data.

## Import Libraries

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.utils import load_object, save_object, plot_confusion_matrix

print('✅ Libraries & Utilities Imported.')

✅ Libraries & Utilities Imported.


## Define IO Functions

In [2]:
# IO operations handled by src.utils
print('✅ IO ready.')

✅ IO ready.


## Load Processed Data (From Step 2)

In [3]:
print('Loading unscaled imputed data...')
X_train_rna_imp = load_object('X_train_rna_imp')
X_test_rna_imp = load_object('X_test_rna_imp')
X_train_meth_imp = load_object('X_train_meth_imp')
X_test_meth_imp = load_object('X_test_meth_imp')
y_train = load_object('y_train')
y_test = load_object('y_test')
rna_feature_names = load_object('rna_feature_names')
meth_feature_names = load_object('meth_feature_names')

print(f'RNA Train Shape:  {X_train_rna_imp.shape}')
print(f'Meth Train Shape: {X_train_meth_imp.shape}')

Loading unscaled imputed data...


RNA Train Shape:  (439, 20155)
Meth Train Shape: (439, 20106)


## 1. Apply Variance Threshold on RNA-Seq
To avoid variance distortion caused by range scaling ((var(x)/(max-min)²)), variance filtering is executed on the raw, unscaled imputed data before MinMax normalization. We filter the lowest 20% variance genes (20th percentile threshold ≈ 22.6). All canonical PAM50 biomarkers (ESR1, ERBB2, PGR, FOXA1, MKI67) are verified to survive. MinMax scaling is applied strictly afterwards.

In [4]:
print('Applying Variance Threshold on Unscaled RNA Data (Bottom 20% filtered)...')
var_rna = np.var(X_train_rna_imp, axis=0)
thresh_rna = float(np.percentile(var_rna, 20))

sel_var_rna = VarianceThreshold(threshold=thresh_rna)
X_train_rna_vraw = sel_var_rna.fit_transform(X_train_rna_imp)
X_test_rna_vraw = sel_var_rna.transform(X_test_rna_imp)
rna_var_indices = sel_var_rna.get_support(indices=True)
rna_var_names = [rna_feature_names[i] for i in rna_var_indices]

# Verify survival of canonical PAM50 biomarkers
canonical_pam50 = ['ESR1', 'ERBB2', 'PGR', 'FOXA1', 'MKI67']
surv_set = set(rna_var_names)
print('Canonical PAM50 Gene Survival Check (Unscaled Variance Filter):')
for g in canonical_pam50:
    print(f'  - {g:7s}: {"✅ Survived" if g in surv_set else "❌ Dropped"}')

# MinMax scaling applied strictly AFTER variance filtering
scaler_rna = MinMaxScaler()
X_train_rna_var = scaler_rna.fit_transform(X_train_rna_vraw)
X_test_rna_var = scaler_rna.transform(X_test_rna_vraw)
print(f'✅ RNA Variance Filtering Done. Kept: {X_train_rna_var.shape[1]}/{X_train_rna_imp.shape[1]}')

Applying Variance Threshold on Unscaled RNA Data (Bottom 20% filtered)...
Canonical PAM50 Gene Survival Check (Unscaled Variance Filter):
  - ESR1   : ✅ Survived
  - ERBB2  : ✅ Survived
  - PGR    : ✅ Survived
  - FOXA1  : ✅ Survived
  - MKI67  : ✅ Survived
✅ RNA Variance Filtering Done. Kept: 16124/20155


## 2. Apply Variance Threshold on Methylation
Similarly, variance thresholding is applied to raw, unscaled methylation beta values (filtering the bottom 20% variance, threshold ≈ 0.0004) before MinMax scaling.

In [5]:
print('Applying Variance Threshold on Unscaled Methylation Data (Bottom 20% filtered)...')
var_meth = np.var(X_train_meth_imp, axis=0)
thresh_meth = float(np.percentile(var_meth, 20))

sel_var_meth = VarianceThreshold(threshold=thresh_meth)
X_train_meth_vraw = sel_var_meth.fit_transform(X_train_meth_imp)
X_test_meth_vraw = sel_var_meth.transform(X_test_meth_imp)
meth_var_indices = sel_var_meth.get_support(indices=True)
meth_var_names = [meth_feature_names[i] for i in meth_var_indices]

scaler_meth = MinMaxScaler()
X_train_meth_var = scaler_meth.fit_transform(X_train_meth_vraw)
X_test_meth_var = scaler_meth.transform(X_test_meth_vraw)
print(f'✅ Methylation Variance Filtering Done. Kept: {X_train_meth_var.shape[1]}/{X_train_meth_imp.shape[1]}')

Applying Variance Threshold on Unscaled Methylation Data (Bottom 20% filtered)...


✅ Methylation Variance Filtering Done. Kept: 16085/20106


## Methodological Note: Supervised Validation in the p >> n Regime
In Phase 1, approximately 32,208 features remain for 439 training samples (p >> n). In this extreme high-dimensional regime, unregularized LDA has a singular within-class covariance matrix, resulting in pseudo-inverse failure and misleadingly low accuracy (~31%, worse than the 53.6% majority baseline). Crucially, evaluating the test set at this intermediate stage would cause severe data snooping/leakage. Therefore, test evaluation is strictly omitted here, and supervised intermediate evaluations begin in Phase 2 using regularized shrinkage LDA on training CV only.

In [6]:
print('Methodological Note on Intermediate Validation in Phase 1:')
print('With p=32,208 features and n=439 samples (p >> n), unregularized LDA has singular covariance')
print('and performs worse than chance (31% accuracy vs 53.6% majority baseline).')
print('Evaluating X_test at this stage causes test data leakage. Hence, test evaluation is removed,')
print('and supervised intermediate evaluations begin in Phase 2 using shrinkage LDA on training CV only.')

Methodological Note on Intermediate Validation in Phase 1:
With p=32,208 features and n=439 samples (p >> n), unregularized LDA has singular covariance
and performs worse than chance (31% accuracy vs 53.6% majority baseline).
Evaluating X_test at this stage causes test data leakage. Hence, test evaluation is removed,
and supervised intermediate evaluations begin in Phase 2 using shrinkage LDA on training CV only.


## Saving Result
Saving the reduced datasets for the next step (ANOVA/ReliefF).

In [7]:
print("Saving variance-filtered datasets to '../outputs/'...")

save_object(X_train_rna_var, 'X_train_rna_var')
save_object(X_test_rna_var,  'X_test_rna_var')
save_object(rna_var_indices, 'feat_indices_rna_var')
save_object(rna_var_names, 'feat_names_rna_var')

save_object(X_train_meth_var, 'X_train_meth_var')
save_object(X_test_meth_var,  'X_test_meth_var')
save_object(meth_var_indices, 'feat_indices_meth_var')
save_object(meth_var_names, 'feat_names_meth_var')

print('\n🎉 Feature Selection Step 1 (Variance Thresholding) Completed Successfully.')

Saving variance-filtered datasets to '../outputs/'...


💾 Saved: ../outputs\X_train_rna_var.npz (+ .pkl)
💾 Saved: ../outputs\X_test_rna_var.npz (+ .pkl)
💾 Saved: ../outputs\feat_indices_rna_var.npz (+ .pkl)
💾 Saved: ../outputs\feat_names_rna_var.pkl


💾 Saved: ../outputs\X_train_meth_var.npz (+ .pkl)
💾 Saved: ../outputs\X_test_meth_var.npz (+ .pkl)
💾 Saved: ../outputs\feat_indices_meth_var.npz (+ .pkl)
💾 Saved: ../outputs\feat_names_meth_var.pkl

🎉 Feature Selection Step 1 (Variance Thresholding) Completed Successfully.
